In [1]:
import numpy as np

In [2]:
#Método de Jacobi
def jacobi(A,b,IteMax,erro):
  lenght = len(b)
  #L é triangular inferior, D é matriz diagonal e U é triangular superior
  L = np.zeros((lenght, lenght))
  D = np.zeros((lenght, lenght))
  U = np.zeros((lenght, lenght))

  for i in range(lenght):
    for j in range(lenght):
      if i>j:
        L[i,j] = A[i,j]
      elif i==j:
        D[i,j] = A[i,j]
      else:
        U[i,j] = A[i,j]

  x = np.zeros(lenght)
  x_novo = np.zeros(lenght)
  inversa_D = np.linalg.inv(D)
  erro_absoluto = 1
  for i in range(IteMax):
      x_novo = inversa_D @ (b - (L+U)@x)
      #Usando erro relativo
      erro_relativo = np.linalg.norm(x_novo - x)/np.linalg.norm(x_novo)

      if erro_relativo < erro:
        print(f"Convergiu após {i+1} iterações")
        return x_novo

      x = x_novo
  print("Número de iterações excedidas")
  return x

matriz_A = np.array([[10,2.,1],[1,5.,1],[2,3,10.]])
vetor_b = np.array([7,-8,6.])
erro = 10**-2
maxIte = 30

print(jacobi(matriz_A.copy(),vetor_b.copy(),maxIte,erro))


Convergiu após 5 iterações
[ 0.99792 -1.99956  0.99676]


In [12]:
#Método de Gauss-Seidel
def gauss_seidel(A,b,IteMax,erro):
  lenght = len(b)
  x = np.zeros(lenght)
  x_novo = np.zeros(lenght)
  L_star = np.zeros((lenght,lenght))
  U_star = np.zeros((lenght,lenght))
  Identidade = np.identity(lenght)
  b_star = np.zeros((lenght))

  for i in range (lenght):
    for j in range(lenght):
      if i>j:
        L_star[i,j] = A[i,j]/A[i,i]
      elif i<j:
        U_star[i,j] = A[i,j]/A[i,i]

    b_star[i] = b[i]/A[i,i]

  #(L* + I)**-1
  inv_Le_I = np.linalg.inv(L_star + Identidade)
  for i in range(IteMax):
    x_novo = ((inv_Le_I)*-1) @ U_star @ x + inv_Le_I @ b_star

    erro_at = np.linalg.norm(x_novo-x)/max(np.linalg.norm(x_novo), 1e-15)
    if erro_at<erro:
      print(f"Convergiu com {i+1} iterações")
      return x_novo

    x = x_novo
  print("Número de iterações excedidas")
  return x

matriz_A = np.array([[5,1.,1],[3,4.,1],[3,3,6.]])
vetor_b = np.array([5,6,0.])
erro = 10**-2
maxIte = 30

print(gauss_seidel(matriz_A.copy(),vetor_b.copy(),maxIte,erro))


Convergiu com 4 iterações
[ 1.001625  0.998625 -1.000125]


In [38]:
#Método de SOR
def SOR(A,b,IteMax,erro,w):
  lenght = len(b)
  L_star = np.zeros((lenght,lenght))
  U_star = np.zeros((lenght,lenght))
  Identidade = np.identity(lenght)
  b_star = np.zeros((lenght))

  A_star = np.zeros((lenght,lenght))
  xGS = np.zeros(lenght)
  xSOR = np.zeros(lenght)
  x = np.zeros(lenght)

  #Inicializando
  for i in range (lenght):
    for j in range(lenght):
      if i>j:
        L_star[i,j] = A[i,j]/A[i,i]
      if i<j:
        U_star[i,j] = A[i,j]/A[i,i]

    b_star[i] = b[i]/A[i,i]


  #Usa o xGS como chute inicial
  inv_Le_I = np.linalg.inv(L_star + Identidade)
  for i in range(IteMax):
    x = xGS.copy()
    xGS = ((inv_Le_I) *-1) @ U_star @ xGS + inv_Le_I @ b_star

    erro_absoluto = np.linalg.norm(xGS - x)
    if erro_absoluto<erro:
      break

  #SOR
  xSOR = xGS
  A_star = L_star + Identidade + U_star
  for k in range(IteMax):
    x = xSOR.copy()
    for i in range(lenght):
      soma1=0.0
      for j in range(i):
        soma1 += A_star[i,j]*xSOR[j]

      soma2=0.0
      for j in range(i+1,lenght):
        soma2 += A_star[i,j]*xSOR[j]

      xSOR[i] = (1-w)*xSOR[i] + w*(b_star[i]-soma1-soma2)

    #Erro Relativo
    erro_at = np.linalg.norm(xSOR-x)/max(np.linalg.norm(xSOR),np.linalg.norm(x))
    if erro_at < erro:
      print(f"Convergiu apos {k+1} iterações")
      return xSOR

  print("Númeor de iterações excedidas")
  return xSOR


matriz_A = np.array([[5,1.,1],[3,4.,1],[3,3,6.]])
vetor_b = np.array([5,6,0.])
erro = 10**-2
maxIte = 30
w = 1.2
print(SOR(matriz_A.copy(),vetor_b.copy(),maxIte,erro,w))


Convergiu apos 1 iterações
[ 1.000035   1.000281  -1.0001646]


In [37]:
def SOR_chute0(A,b,IteMax,erro,w):
  lenght = len(b)
  L_star = np.zeros((lenght,lenght))
  U_star = np.zeros((lenght,lenght))
  Identidade = np.identity(lenght)
  b_star = np.zeros((lenght))

  xSOR = np.zeros(lenght)
  A_star = np.zeros((lenght,lenght))
  x = np.zeros(lenght)

  #Inicializando
  for i in range (lenght):
    for j in range(lenght):
      if i>j:
        L_star[i,j] = A[i,j]/A[i,i]
      if i<j:
        U_star[i,j] = A[i,j]/A[i,i]

    b_star[i] = b[i]/A[i,i]



  #SOR
  #Usando chute inical como 0
  A_star = L_star + Identidade + U_star
  for k in range(IteMax):
    x = xSOR.copy()
    for i in range(lenght):
      soma1=0.0
      for j in range(i):
        soma1 += A_star[i,j]*xSOR[j]

      soma2=0.0
      for j in range(i+1,lenght):
        soma2 += A_star[i,j]*xSOR[j]

      xSOR[i] = (1-w)*xSOR[i] + w*(b_star[i]-soma1-soma2)

      #Erro Relativo
    erro_at = np.linalg.norm(xSOR-x)/max(np.linalg.norm(xSOR),np.linalg.norm(x))
    if erro_at < erro:
      print(f"Convergiu apos {k+1} iterações")
      return xSOR

  print("Númeor de iterações excedidas")
  return xSOR

matriz_A = np.array([[5,1.,1],[3,4.,1],[3,3,6.]])
vetor_b = np.array([5,6,0.])
erro = 10**-2
maxIte = 30
w = 1.2
print(SOR_chute0(matriz_A.copy(),vetor_b.copy(),maxIte,erro,w))

Convergiu apos 5 iterações
[ 1.00075136  0.99939459 -1.00044834]


In [36]:
#Exercício
matriz_A = np.array([[5,1.,1],[3,4.,1],[3,3,6.]])
vetor_b = np.array([5,6,0.])
erro = 10**-2
maxIte = 10
w = 1.2
print(jacobi(matriz_A.copy(),vetor_b.copy(),maxIte,erro))
print(gauss_seidel(matriz_A.copy(),vetor_b.copy(),maxIte,erro))
print(SOR_chute0(matriz_A.copy(),vetor_b.copy(),maxIte,erro,w))


Número de iterações excedidas
[ 0.98647363  0.97697803 -1.02563135]
Convergiu com 4 iterações
[ 1.001625  0.998625 -1.000125]
Convergiu apos 5 iterações
[ 1.00075136  0.99939459 -1.00044834]
